In [ ]:
import json
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from helpers.triplets import TripletGenerator, transform_triplets_for_embedding, triplet_to_natural_language
from llama_index.llms.openai import OpenAI
from llama_index.core import PropertyGraphIndex
from llama_index.core.graph_stores import SimplePropertyGraphStore
from llama_index.core import Settings
from IPython.display import Markdown, display
from dotenv import load_dotenv
from llama_index.core import StorageContext
from llama_index.core.schema import TextNode
from typing import List, Tuple, Dict
import os
from tqdm import tqdm
from datetime import datetime
import re

load_dotenv("devops/env/default.env")
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

with open('btc_blocks.json', 'r') as f:
    blocks_data = json.load(f)

with open('economic_indicators.json', 'r') as f:
    economic_data = json.load(f)

with open('on_chain_metrics.json', 'r') as f:
    onchain_data = json.load(f)


tg = TripletGenerator()
triplets = tg.load_and_process_data(blocks_data, economic_data, onchain_data)

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
llm = OpenAI(model="gpt-3.5-turbo-instruct", 
             temperature=0,
             api_key=OPENAI_API_KEY)
Settings.llm = llm
Settings.embed_model = embed_model

"""
graph_store = SimpleGraphStore()
storage_context = StorageContext.from_defaults(graph_store=graph_store)

kg_index = KnowledgeGraphIndex(
    [],
    storage_context=storage_context,
    include_embeddings=True,
)
"""
# The existing upsert_triplets performs sequential insertion including sequential embedding generation which is verrryyy slow
def batch_upsert_triplets_and_nodes(kg: KnowledgeGraphIndex, triplets: List[Tuple[Tuple[str, str, str], Dict]], include_embeddings: bool = True) -> None:
    """Batch insert triplets with their embeddings."""
    # First insert all triplets to the graph store
    for (subj, pred, obj), metadata in triplets:
        # Add the triplet to the graph store
        kg._graph_store.upsert_triplet(subj, pred, obj)

        # Create a node that represents this triplet with its metadata
        triplet_text = triplet_to_natural_language((subj, pred, obj))
        node_id = f"{subj}_{pred}_{obj}"
        
        # Create a node with the metadata
        triplet_node = TextNode(
            text=triplet_text,
            metadata=metadata,
            id_=node_id
        )
        kg.add_node([subj, obj], triplet_node)
    
    if include_embeddings:
        # Generate triplet strings for embedding
        triplet_strs = transform_triplets_for_embedding([triplet for triplet, _ in triplets])
        
        # Get embeddings in batch
        batch_embeddings = kg._embed_model.get_text_embedding_batch(triplet_strs)
        
        # Add embeddings to the index structure
        for triplet_str, embedding in tqdm(zip(triplet_strs, batch_embeddings)):
            kg._index_struct.add_to_embedding_dict(triplet_str, embedding)
            
        # Update the storage
        kg._storage_context.index_store.add_index_struct(kg._index_struct)


'\ngraph_store = SimpleGraphStore()\nstorage_context = StorageContext.from_defaults(graph_store=graph_store)\n\nkg_index = KnowledgeGraphIndex(\n    [],\n    storage_context=storage_context,\n    include_embeddings=True,\n)\n'

In [5]:
graph_store = SimplePropertyGraphStore()
kg_index = PropertyGraphIndex(
    [],
    property_graph_store=graph_store
)


## Core Domain Entities

1. (Bitcoin, IS_A, BLOCKCHAIN)
2. (Bitcoin, CREATED_ON, "2009-01-03")
3. (S&P500, IS_A, ECONOMIC_INDICATOR)
4. (Federal_Funds_Rate, IS_A, ECONOMIC_INDICATOR)
5. (Hash_Rate, IS_A, NETWORK_METRIC)
6. (Hash_Rate, HAS_UNIT, "TH/s")
7. (Transaction_Volume, IS_A, NETWORK_METRIC)
8. (Transaction_Volume, HAS_UNIT, "BTC")
9. (Mempool_Size, IS_A, NETWORK_METRIC)
10. (Mempool_Size, HAS_UNIT, "bytes")

## Block Entities (Backbone)

11. (Block_894214, IS_A, BLOCK)
12. (Block_894214, HAS_HEIGHT, 894214)
13. (Block_894214, HAS_HASH, "00000000000000000000ab2d5af936a7c0b91f4216ac3a3e7f0ba89d9f20e03f")
18. (Block_894214, HAS_DIFFICULTY, 121507793131897.94)
19. (Block_894214, HAS_TRANSACTION_COUNT, 2453)

## Time Series Datapoints

21. (HashRate_20250402, IS_A, DATAPOINT)
22. (HashRate_20250402, HAS_VALUE, 972645626.88)
27. (Hash_Rate, HAS_DATAPOINT, HashRate_20250402)
28. (MempoolSize_20250331, IS_A, DATAPOINT)
29. (MempoolSize_20250331, HAS_VALUE, 6381844.5)
31. (Mempool_Size, HAS_DATAPOINT, MempoolSize_20250331)
32. (SP500_20250418, IS_A, DATAPOINT)
33. (SP500_20250418, HAS_VALUE, 5823.45)
35. (S&P500, HAS_DATAPOINT, SP500_20250418)

## Economic Context Connections

36. (Block_894214, HAS_ECONOMIC_CONTEXT, SP500_20250418)
37. (Block_894214, HAS_ECONOMIC_CONTEXT, FedRate_20250418)
38. (FedRate_20250418, HAS_VALUE, 5.25)
40. (Federal_Funds_Rate, HAS_DATAPOINT, FedRate_20250418)
41. (Block_894214, MINED_DURING_RATE, 5.25)

## Derived Relationships

42. (Hash_Rate, CORRELATES_WITH, Federal_Funds_Rate)
43. (Hash_Rate, CORRELATION_COEFFICIENT, -0.68)
44. (Transaction_Volume, CORRELATES_WITH, S&P500)
45. (Transaction_Volume, CORRELATION_COEFFICIENT, 0.72)
46. (Bitcoin_Price, INFLUENCED_BY, Federal_Funds_Rate)
47. (Bitcoin_Price, INFLUENCE_DIRECTION, "negative")
48. (CorrelationEvent_20250418, CONNECTS, Bitcoin_Price)
49. (CorrelationEvent_20250418, CONNECTS, Federal_Funds_Rate)
50. (CorrelationEvent_20250418, OBSERVED_ON, "2025-04-18")

## Transactions and UTXOs

51. (Transaction_a72f4d, IS_A, TRANSACTION)
52. (Transaction_a72f4d, HAS_TXID, "a72f4d8b6e9c3f2a1d5e8b7c6d3f2a1d5e8b7c6d3f2a1d5")
53. (Transaction_a72f4d, INCLUDED_IN, Block_894214)
54. (Transaction_a72f4d, HAS_FEE, 0.00025)
55. (Transaction_a72f4d, HAS_INPUT_COUNT, 3)
56. (Transaction_a72f4d, HAS_OUTPUT_COUNT, 2)
57. (Block_894214, CONTAINS, Transaction_a72f4d)
58. (Address_bc1q4, INVOLVED_IN, Transaction_a72f4d)
59. (Address_bc1q4, IS_A, ADDRESS)

## Time Period Analysis

60. (April_2025, IS_A, TIME_PERIOD)
61. (April_2025, HAS_YEAR, 2025)
62. (April_2025, HAS_MONTH, 4)
63. (Block_894214, MINED_DURING, April_2025)
64. (April_2025, AVG_HASH_RATE, 983456789.25)
65. (April_2025, AVG_TRANSACTION_VOLUME, 256789.34)
